# Argus — Dataset Creation (CNN+LSTM Windowed Image Sequences)

The fifth model family: a hybrid `TimeDistributed(CNN) → LSTM` that sees an actual windowed
*sequence* of face-crop images (curated durations from 3s up to 20s — see "Pipeline Configuration
Constants" below — related to but deliberately not identical to
[`01_dataset_creation_lstm.ipynb`](./01_dataset_creation_lstm.ipynb)'s own multi-duration
sliding-window scheme), rather than either a single cropped image (`06`/`07`'s CNN) or
hand-engineered geometric features (the LSTM). This is the most complete fix for the
single-instant-in-time limitation that capped every other single-frame model around ~40%: it
combines raw-pixel visual cues (which plain geometric ratios can't see) with genuine temporal
context (which a single image can't see either).

**Calibrated expectation, stated plainly up front:** this is also, by a wide margin, the most
data-hungry model built so far — a deep model over raw pixel *sequences*, trained on ~54
subjects. Real overfitting risk here is higher than anywhere else in this project, not lower.
Worth trying — it's the theoretically strongest architecture for this problem — but don't assume
"more complete architecture" automatically means "better result" on a dataset this size; treat it
as a real experiment with a real chance of underperforming the LSTM, not a foregone upgrade.

**Reads:** `dataset_processed/face_crops_index.csv` and the `.jpg` files it points to, written by
[`06_dataset_creation_face_crops.ipynb`](./06_dataset_creation_face_crops.ipynb) — **run that
first if you haven't** (it hasn't produced output yet as of this notebook being written). This
notebook does **not** re-extract or re-crop anything from video — it only builds a lightweight
window *index* (frame-range references into crops that already exist), the same principle that
turned the LSTM's dataset from thousands of `.npy` files into one CSV: reuse what's already
extracted, don't multiply storage by re-saving overlapping copies of it.

**Writes:** `dataset_processed/cnn_lstm_windows_index.csv` — one row per valid window, storing an
ordered, `;`-joined list of that window's real crop image paths (not the pixels themselves; those
are decoded lazily by `10_cnn_lstm_training.ipynb`'s `tf.data` pipeline, same lazy-loading pattern
`07_cnn_training.ipynb` already uses for single images). This means dataset *creation* here never
loads or accumulates pixel data at all — regardless of window count or duration, this notebook
only ever holds path strings in memory, so it doesn't have a RAM-crash risk to design around the
way the geometric LSTM's `MAX_TIMESTEPS` did (see `01_dataset_creation_lstm.ipynb`'s note on why
`MAX_TIMESTEPS` was brought down from 120 to 60 — that was about accumulating flattened *feature*
rows in memory, a failure mode this path-only index doesn't share).


**Update:** each window row also now carries a `geometric_feature_seq` column -- a parallel,
`;`-joined-per-frame EAR/MAR/blendshape feature vector alongside `image_paths` -- so
`10_cnn_lstm_training.ipynb`'s model can fuse hand-engineered geometric ratios with the raw-pixel
CNN embedding at each timestep. See the "Geometric Features for Fusion" section below for why and
how these are computed (directly on the saved crops, via a fifth copy of
`GeometricRatioFeatureLayer`) without needing to re-run `06_dataset_creation_face_crops.ipynb`.



## Google Drive Connection & Project Setup


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

project_folder = "/content/drive/MyDrive/Argus"
print(f"Google Drive successfully mounted! Base project directory: {project_folder}")


In [ ]:
import os

models_folder = f"{project_folder}/models"
dataset_folder = f"{project_folder}/dataset"
processed_folder = f"{dataset_folder}/dataset_processed"
face_crops_folder = f"{processed_folder}/face_crops"
face_crops_index_csv_path = os.path.join(processed_folder, "face_crops_index.csv")

if not os.path.exists(face_crops_index_csv_path):
    raise FileNotFoundError(
        f"'{face_crops_index_csv_path}' not found. Run 06_dataset_creation_face_crops.ipynb "
        "first -- this notebook only indexes windows over crops it produces, it doesn't extract "
        "any crops itself."
    )

print("Project folder structure validated.")


In [ ]:
# Drowsiness class labels -- must match what 06_dataset_creation_face_crops.ipynb wrote into
# face_crops_index.csv (this notebook inherits the label from there, it does not re-map).
CLASS_NAMES = ["Not Drowsy", "Drowsy"]
NUM_CLASSES = len(CLASS_NAMES)
print(f"{NUM_CLASSES} classes: {CLASS_NAMES}")

## Pipeline Configuration Constants

**Curated window durations (3s/5s/10s/20s), not the geometric LSTM's dense 1s-step sweep.**
`01_dataset_creation_lstm.ipynb` builds one window config per second from 1-6s (six configs).
This notebook instead uses a small curated set spanning a wider 3-20s range — fewer, more
distinct durations means less near-duplicate redundancy between adjacent configs (adjacent-second
durations would mostly overlap in content — a 3s and a 4s window share the great majority of
their frames), and keeps the window index a manageable size even at the wider range.

**`sampling_fps = 5`, matching `06_dataset_creation_face_crops.ipynb`'s actual rate — not 10, and
no longer 1.** An earlier version of this notebook hardcoded `sampling_fps = 10` under the
assumption that 06 sampled that densely; a later version corrected it to `1`. 06 now samples at
**5 FPS** — raised from that `1` to give the data-hungry CNN / CNN+LSTM path substantially more
crops per clip (see 06's own "Pipeline Configuration Constants" section for the volume-vs-
redundancy tradeoff). Getting this wrong doesn't raise an error: window sizing below is
`duration_sec * sampling_fps`, so a wrong `sampling_fps` silently produces windows of the wrong
real-world duration while still looking internally consistent. Keep this in sync with 06's real
constant, not a guess.

**`MAX_TIMESTEPS_IMG = 100`, not 20 or 60.** The geometric LSTM's dataset pads to a fixed
`max_context_sec * sampling_fps`; here that's `20 * 5 = 100`. Each padded slot is a full
`(IMG_SIZE, IMG_SIZE, 3)` image tensor rather than a 58-float feature row, so padding/batch cost
scales with tensor size — a 100-slot image sequence per sample is a real memory cost in
`10_cnn_lstm_training.ipynb`'s data pipeline (5x the old 20-slot version at 1 FPS), not free
headroom the way extra slots are for a feature-row sequence. This is still images-per-slot, not
feature-vectors-per-slot — keep that distinction when reasoning about the cost.


In [ ]:
sampling_fps = 5                       # Must match 06_dataset_creation_face_crops.ipynb's actual
                                        # sampling_fps (now 5 FPS, raised from an earlier 1 FPS
                                        # for more CNN training volume) -- NOT the geometric
                                        # LSTM's 10 FPS. Getting this wrong doesn't error -- it
                                        # silently makes every window's real duration wrong,
                                        # since window sizing below is `duration_sec * sampling_fps`.
window_configs = [3.0, 5.0, 10.0, 20.0]  # Curated durations, not a dense 1s-step sweep like
                                          # 01_dataset_creation_lstm.ipynb's -- fewer, more
                                          # distinct configs means less near-duplicate redundancy
                                          # between adjacent durations.
min_context_sec = min(window_configs)   # 3
max_context_sec = max(window_configs)   # 20

MAX_TIMESTEPS_IMG = int(max_context_sec * sampling_fps)  # 20 * 5 = 100 -- a full image per slot;
                                                          # see note above on why this is a real
                                                          # memory cost in 10, not free headroom.

print(f"✅ Pipeline constants initialized. Windows: {window_configs}s, MAX_TIMESTEPS_IMG={MAX_TIMESTEPS_IMG}")


## Geometric Features for Fusion (EAR/MAR/Blendshapes)

Alongside each window's face-crop images, this notebook now also extracts a small per-frame
**geometric feature vector** -- the same family of EAR/MAR/blendshape features the geometric
LSTM (`01_dataset_creation_lstm.ipynb`/`03_model_training_lstm.ipynb`) is built on -- so
`10_cnn_lstm_training.ipynb`'s model can fuse hand-engineered ratios with raw-pixel CNN
embeddings per timestep, instead of relying on pixels alone. Motivation: the root `CLAUDE.md`'s
"Model results and current status" already shows these features carry real (if individually
weak) signal -- `MAR` ranks #4 by Spearman |r| against the drowsiness label in
`02_dataset_creation_flat.ipynb`'s own correlation cell, and several eye/brow blendshapes rank
above it -- and a CNN embedding has no built-in way to reconstruct that ratio-normalized signal
from raw pixels on its own. (Those specific ranks and |r| values were measured on an earlier version of this feature set;
the 10-feature fusion set below is a fixed curated list, not re-derived per run.)

**Ten curated features, not all 58:** `EAR_left`, `EAR_right`, `MAR` (explicitly requested,
alongside the top-ranked blendshapes from `02`'s actual printed Spearman ranking --
`eyeWideRight`, `eyeBlinkRight`, `browOuterUpRight`, `eyeBlinkLeft`, `eyeSquintLeft`,
`eyeWideLeft`, `eyeSquintRight`). **This list intentionally does not reuse
`04_random_forest_training.ipynb`'s curated-7 list** (`eyeBlinkLeft/Right`, `EAR_mean`,
`eyeSquintLeft/Right`, `jawOpen`, `pitch`) -- cross-checking that list against `02`'s own printed
correlation ranking shows `jawOpen` (|r|=0.048) and `pitch` (|r|=0.013) are actually near the
*bottom* of the ranking, not top features; this notebook selects from the real ranking instead of
propagating that inconsistency into a fifth place. `EAR_left`/`EAR_right` are weak by this same
ranking (|r| ~0.03-0.05) but kept anyway since they were explicitly requested and are the feature
the rest of the project's geometric pipeline is built around.

**Computed on the saved crop image, not the original frame.** `06_dataset_creation_face_crops.ipynb`
only ever ran the lightweight Face Detector (`BlazeFace`, bbox-only) -- it never ran
`FaceLandmarker`, so no landmarks exist yet for these frames. Re-running `06` against all ~54
subjects just to add `FaceLandmarker` would be an expensive full re-extraction; instead,
`FaceLandmarker` runs here, once, directly on each already-saved crop `.jpg`, in `IMAGE` (not
`VIDEO`) running mode, since crops are handled independently rather than as a timestamped stream.
**Caveat worth stating plainly:** EAR/MAR/blendshape ratios are computed from landmark
*distances relative to each other*, so they're largely scale/crop-invariant and should transfer
reasonably from a tight crop to a full frame -- but this is still a different input than what
`01`/`02`/deployment compute these same ratios from (the full original frame), so treat this as a
reasonable approximation, not a guaranteed match to the geometric LSTM's own features.

**A fifth place `GeometricRatioFeatureLayer` must now be kept byte-identical.** The root
`CLAUDE.md` already tracks four copies (`01_dataset_creation_lstm.ipynb`,
`02_dataset_creation_flat.ipynb`, `08_deployment_export_lstm.ipynb`,
`src/cv-argus/src/model/layers.py`); the copy below is a fifth and needs the same manual-sync
discipline -- there is still no automated check.


In [ ]:
import os
import urllib.request

media_pipe_url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task"
media_pipe_filename = "face_landmarker.task"
media_pipe_path = os.path.join(models_folder, media_pipe_filename)

# Same idempotent download pattern as 01_dataset_creation_lstm.ipynb/02_dataset_creation_flat.ipynb
# -- one of those two has very likely already downloaded this exact file into models_folder, so
# this is normally a no-op skip, not a fresh download.
if not os.path.exists(media_pipe_path):
    print(f"Downloading MediaPipe Face Landmarker model to {media_pipe_path}...")
    try:
        urllib.request.urlretrieve(media_pipe_url, media_pipe_path)
        print("Download complete.")
    except Exception as e:
        raise RuntimeError(f"Failed to download the MediaPipe model from {media_pipe_url}. Error: {e}")
else:
    print(f"Model already exists at {media_pipe_path}. Skipping download.")

import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

# IMAGE (not VIDEO) running mode -- unlike 01/02, which process a continuous timestamped video
# stream, here each crop is an independent still image with no meaningful inter-frame timestamp
# relationship to any other crop.
geo_base_options = mp_python.BaseOptions(model_asset_path=os.path.abspath(media_pipe_path))
geo_face_landmarker_options = vision.FaceLandmarkerOptions(
    base_options=geo_base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=True,
)

print("FaceLandmarker (IMAGE mode) configuration ready for per-crop geometric feature extraction.")


In [ ]:
import tensorflow as tf
import numpy as np

# --- Sync warning: byte-identical copy of GeometricRatioFeatureLayer -- see the markdown cell
# above. Keep this in sync with the other four copies (01_dataset_creation_lstm.ipynb,
# 02_dataset_creation_flat.ipynb, 08_deployment_export_lstm.ipynb,
# src/cv-argus/src/model/layers.py) if it is ever changed anywhere. ---
@tf.keras.utils.register_keras_serializable(package="Argus")
class GeometricRatioFeatureLayer(tf.keras.layers.Layer):
    """
    TensorFlow-native counterpart to compute_ear, compute_mar, and rotation_matrix_to_euler.
    Inputs:
        landmarks_xy : (batch, 478, 2)   -- normalized (x, y) coordinates
        rotation_matrix : (batch, 3, 3)  -- top-left 3x3 block of the facial transformation matrix
    Returns:
        (batch, 7) : [EAR_left, EAR_right, MAR, pitch, yaw, roll, ear_mar_valid]
    """
    def __init__(self, pose_validity_threshold_deg=20.0, **kwargs):
        super().__init__(**kwargs)
        self.pose_validity_threshold_deg = pose_validity_threshold_deg
        self.left_eye_idx  = tf.constant([33, 160, 158, 133, 153, 144], dtype=tf.int32)
        self.right_eye_idx = tf.constant([362, 385, 387, 263, 373, 380], dtype=tf.int32)
        self.mouth_idx     = tf.constant([61, 291, 13, 14], dtype=tf.int32)
        self.blendshape_names = [
            "browDownLeft", "browDownRight", "browInnerUp", "browOuterUpLeft", "browOuterUpRight",
            "cheekPuff", "cheekSquintLeft", "cheekSquintRight", "eyeBlinkLeft", "eyeBlinkRight",
            "eyeLookDownLeft", "eyeLookDownRight", "eyeLookInLeft", "eyeLookInRight",
            "eyeLookOutLeft", "eyeLookOutRight", "eyeLookUpLeft", "eyeLookUpRight",
            "eyeSquintLeft", "eyeSquintRight", "eyeWideLeft", "eyeWideRight",
            "jawForward", "jawLeft", "jawOpen", "jawRight", "mouthClose", "mouthDimpleLeft",
            "mouthDimpleRight", "mouthFrownLeft", "mouthFrownRight", "mouthFunnel",
            "mouthLeft", "mouthLowerDownLeft", "mouthLowerDownRight", "mouthPressLeft",
            "mouthPressRight", "mouthPucker", "mouthRight", "mouthRollLower",
            "mouthRollUpper", "mouthShrugLower", "mouthShrugUpper", "mouthSmileLeft",
            "mouthSmileRight", "mouthStretchLeft", "mouthStretchRight", "mouthUpperUpLeft",
            "mouthUpperUpRight", "noseSneerLeft", "noseSneerRight"
        ]

    def get_config(self):
        config = super().get_config()
        config.update({"pose_validity_threshold_deg": self.pose_validity_threshold_deg})
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

    @staticmethod
    def _dist(points, i, j):
        a = points[..., i, :]
        b = points[..., j, :]
        return tf.norm(a - b, axis=-1)

    def _ear(self, landmarks, idx):
        p = tf.gather(landmarks, idx, axis=-2)
        vertical = self._dist(p, 1, 5) + self._dist(p, 2, 4)
        horizontal = 2.0 * self._dist(p, 0, 3)
        return tf.math.divide_no_nan(vertical, horizontal)

    def _mar(self, landmarks, idx):
        p = tf.gather(landmarks, idx, axis=-2)
        vertical = self._dist(p, 2, 3)
        horizontal = self._dist(p, 0, 1)
        return tf.math.divide_no_nan(vertical, horizontal)

    @staticmethod
    def _rotation_matrix_to_euler(R):
        r00, r10, r20 = R[..., 0, 0], R[..., 1, 0], R[..., 2, 0]
        r21, r22 = R[..., 2, 1], R[..., 2, 2]
        r11, r12 = R[..., 1, 1], R[..., 1, 2]
        sy = tf.sqrt(r00**2 + r10**2)
        singular = sy < 1e-6
        pitch = tf.where(singular, tf.atan2(-r12, r11), tf.atan2(r21, r22))
        yaw = tf.atan2(-r20, sy)
        roll = tf.where(singular, tf.zeros_like(r10), tf.atan2(r10, r00))
        return pitch * (180.0 / np.pi), yaw * (180.0 / np.pi), roll * (180.0 / np.pi)

    def call(self, landmarks_xy, rotation_matrix):
        ear_left = self._ear(landmarks_xy, self.left_eye_idx)
        ear_right = self._ear(landmarks_xy, self.right_eye_idx)
        mar = self._mar(landmarks_xy, self.mouth_idx)
        pitch, yaw, roll = self._rotation_matrix_to_euler(rotation_matrix)
        ear_mar_valid = tf.cast(tf.logical_and(tf.abs(yaw) < self.pose_validity_threshold_deg, tf.abs(pitch) < self.pose_validity_threshold_deg), tf.float32)
        return tf.stack([ear_left, ear_right, mar, pitch, yaw, roll, ear_mar_valid], axis=-1)


# --- Curated 10-feature fusion set -- EAR/MAR plus the actual top-ranked blendshapes from
# 02_dataset_creation_flat.ipynb's Spearman correlation cell (not 04_random_forest_training.ipynb's
# curated-7, which doesn't match that ranking -- see the markdown cell above). ---
GEO_FEATURE_NAMES = [
    'EAR_left', 'EAR_right', 'MAR',
    'eyeWideRight', 'eyeBlinkRight', 'browOuterUpRight',
    'eyeBlinkLeft', 'eyeSquintLeft', 'eyeWideLeft', 'eyeSquintRight',
]
NUM_GEO_FEATURES = len(GEO_FEATURE_NAMES)

geo_ratio_layer = GeometricRatioFeatureLayer(pose_validity_threshold_deg=20.0)

print(f"Geometric fusion feature set ({NUM_GEO_FEATURES} features): {GEO_FEATURE_NAMES}")


## Building the Window Index

For each clip `(subject, parent_video)`, tile the same 3-20s curated-duration windows over its
**already-cropped** frames (loaded from `face_crops_index.csv`). Windowing is done over
`sample_idx` -- the consecutive sampled-frame position `06` now records -- **not** `frame_idx`
(the raw video frame count, which is spaced by that video's own `frame_stride` and would make
every window spuriously fail a naive contiguity check).

**Tiled per contiguous run, not a single global grid.** A clip's available `sample_idx` values
are first split into maximal *contiguous* runs (a gap opens a new run -- some frames are missing
where the Face Detector found no confident face). Each run is then tiled independently, starting
fresh from that run's own first sample_idx. This -- rather than tiling from a single anchor
(the clip's first available sample) and discarding any window that happens to land on a gap --
avoids wasting real, usable frames: a fixed global grid can leave up to `win_size - 1` real frames
stranded on the wrong side of a gap, off-grid, even though they're perfectly good contiguous data
on their own. In a subject-scarce dataset, every extra window matters.

**Non-overlapping, not sliding.** Within one duration config, windows tile back-to-back
(`stride = win_size` -- the window's own length in samples) rather than sliding by a small fixed
step -- so a run's samples are chopped into consecutive, disjoint 3s chunks, separately disjoint
5s chunks, and so on, with no two windows *of the same duration* sharing a frame. This keeps
subject-grouped train/val/test splitting simple in the training notebook: no near-duplicate
windows straddling a split boundary within a given duration. Windows from *different* duration
configs will still cover overlapping real-time spans of the same clip -- a 3s window and a 10s
window starting at the same point obviously overlap -- but that's inherent to the multi-duration
scheme and intentional (duration diversity in what the model sees), not the kind of overlap this
is avoiding.


In [ ]:
import pandas as pd
import numpy as np

df_face_crops = pd.read_csv(face_crops_index_csv_path)
if 'sample_idx' not in df_face_crops.columns:
    raise ValueError(
        "face_crops_index.csv has no 'sample_idx' column -- re-run 06_dataset_creation_face_crops.ipynb "
        "(it was updated to record this alongside frame_idx; a CSV from before that update won't have it)."
    )
df_face_crops = df_face_crops.sort_values(['subject', 'parent_video', 'sample_idx']).reset_index(drop=True)

print(f"Loaded {len(df_face_crops)} indexed crops across "
      f"{df_face_crops.groupby(['subject', 'parent_video']).ngroups} clips.")

_distinct_levels = sorted(int(x) for x in df_face_crops['level'].unique())
_expected_levels = list(range(1, NUM_CLASSES + 1))
if _distinct_levels != _expected_levels:
    raise ValueError(
        f"face_crops_index.csv has levels {_distinct_levels}, expected exactly {_expected_levels} "
        f"({CLASS_NAMES}). It is stale relative to the binary label scheme -- re-run "
        f"06_dataset_creation_face_crops.ipynb (against dataset/raw_videos_binary/) first. This "
        f"notebook inherits labels from that CSV verbatim and would otherwise emit a mislabeled "
        f"window index."
    )


### Extracting Geometric Features Per Crop

Runs the `FaceLandmarker`/`GeometricRatioFeatureLayer` setup above once per **unique** crop image
referenced by `face_crops_index.csv` (not once per window -- each crop is reused by at most one
window per duration config, but computing it keyed by `image_path` avoids any risk of redundant
work if that ever changes). Builds an in-memory `image_path -> [10 floats]` lookup that
`build_windows_for_clip` below reads from when assembling each window's parallel
`geometric_feature_seq` column, mirroring how it already reads `image_paths` from
`sample_idx_to_path`.


In [ ]:
from tqdm.auto import tqdm
import cv2


def _extract_geo_features(image_path, detector, ratio_layer, feature_names):
    """Runs FaceLandmarker (IMAGE mode) on one saved crop and returns a dict of the named
    geometric features. Falls back to all-zero values (mirroring 02_dataset_creation_flat.ipynb's
    dummy-row convention for a failed detection) if no face is found on the crop -- this should be
    rare, since 06_dataset_creation_face_crops.ipynb already confirmed a confident BlazeFace
    detection on the same image, but a landmarker miss on an already-tight crop isn't impossible.
    """
    bgr = cv2.imread(image_path)
    if bgr is None:
        return {name: 0.0 for name in feature_names}
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_image)

    if not result.face_landmarks:
        return {name: 0.0 for name in feature_names}

    lm = result.face_landmarks[0]
    landmarks_xy = tf.constant([[p.x, p.y] for p in lm], dtype=tf.float32)
    if result.facial_transformation_matrixes:
        R = np.array(result.facial_transformation_matrixes[0])[:3, :3]
    else:
        R = np.eye(3, dtype=np.float32)
    R_tf = tf.constant(R, dtype=tf.float32)

    base_features = ratio_layer(landmarks_xy[tf.newaxis, ...], R_tf[tf.newaxis, ...]).numpy()[0]
    base_dict = dict(zip(['EAR_left', 'EAR_right', 'MAR', 'pitch', 'yaw', 'roll', 'ear_mar_valid'], base_features))

    bs_dict = {b.category_name: b.score for b in result.face_blendshapes[0]} if result.face_blendshapes else {}

    all_features = {**base_dict, **bs_dict}
    return {name: float(all_features.get(name, 0.0)) for name in feature_names}


geo_detector = vision.FaceLandmarker.create_from_options(geo_face_landmarker_options)

geo_features_by_path = {}
n_detection_failures = 0
for image_path in tqdm(df_face_crops['image_path'].unique(), desc="Extracting geometric features per crop"):
    feats = _extract_geo_features(image_path, geo_detector, geo_ratio_layer, GEO_FEATURE_NAMES)
    if all(v == 0.0 for v in feats.values()):
        n_detection_failures += 1
    geo_features_by_path[image_path] = [feats[name] for name in GEO_FEATURE_NAMES]

geo_detector.close()

print(f"Extracted geometric features for {len(geo_features_by_path)} unique crops "
      f"({n_detection_failures} landmark-detection failures, fell back to zeros).")


In [ ]:
def _contiguous_runs(sorted_indices):
    """Split a sorted list of ints into maximal runs of consecutive integers -- e.g.
    [0,1,2,5,6,9] -> [(0,2),(5,6),(9,9)]. A gap in sample_idx (a frame with no confident face
    detection) starts a new run.
    """
    runs = []
    run_start = prev = sorted_indices[0]
    for idx in sorted_indices[1:]:
        if idx != prev + 1:
            runs.append((run_start, prev))
            run_start = idx
        prev = idx
    runs.append((run_start, prev))
    return runs


def build_windows_for_clip(clip_df, window_configs, sampling_fps, geo_features_by_path):
    """clip_df: rows for one (subject, parent_video), already sorted by sample_idx.

    Returns a list of dicts, one per valid window: the ordered image_path list for that window's
    real (contiguous, gap-free) frames -- no padding decided here, that happens at load time in
    10_cnn_lstm_training.ipynb, mirroring how 01_dataset_creation_lstm.ipynb keeps padding out of
    the generation step for everything except the final flattened output. Each window also carries
    a parallel `geometric_feature_seq` -- one GEO_FEATURE_NAMES-ordered, comma-joined feature
    vector per real frame, `;`-joined in the same frame order as image_paths -- see the
    "Geometric Features for Fusion" markdown cell above.

    Windows are non-overlapping within each duration config (stride == window size), tiled
    separately per maximal contiguous run of sample_idx -- see the "Building the Window Index"
    markdown cell above for why tiling per-run (rather than from one global anchor) matters.
    """
    sample_idx_to_path = dict(zip(clip_df['sample_idx'], clip_df['image_path']))
    available_samples = sorted(sample_idx_to_path.keys())
    if not available_samples:
        return []

    runs = _contiguous_runs(available_samples)
    windows = []
    for win_sec in window_configs:
        win_size = int(win_sec * sampling_fps)
        for run_start, run_end in runs:
            # Every index in [start, start + win_size) is guaranteed present -- the run is
            # contiguous by construction, so no gap-check is needed here (unlike a single
            # global-anchor grid, which could land on a gap and require discarding a candidate).
            for start in range(run_start, run_end - win_size + 2, win_size):
                end = start + win_size  # exclusive
                image_paths = [sample_idx_to_path[si] for si in range(start, end)]
                geo_seq = ';'.join(
                    ','.join(f"{v:.6f}" for v in geo_features_by_path[p]) for p in image_paths
                )
                windows.append({
                    'window_duration_sec': win_sec,
                    'n_real_frames': win_size,
                    'start_sample_idx': start,
                    'end_sample_idx': end - 1,
                    'image_paths': ';'.join(image_paths),
                    'geometric_feature_seq': geo_seq,
                })
    return windows


### Running the Window Builder Across All Clips


In [ ]:
from tqdm.auto import tqdm

rows = []
skipped = []

grouped = df_face_crops.groupby(['subject', 'parent_video'], sort=False)

for (subject, parent_video), clip_df in tqdm(grouped, desc="Building windows per clip"):
    level = clip_df['level'].iloc[0]  # constant per clip -- inherited verbatim from 06's face_crops_index.csv, never re-mapped here

    clip_windows = build_windows_for_clip(clip_df, window_configs, sampling_fps, geo_features_by_path)
    if not clip_windows:
        skipped.append((subject, parent_video, "No valid (gap-free) window of any configured duration"))
        continue

    for w in clip_windows:
        w['subject'] = subject
        w['parent_video'] = parent_video
        w['level'] = level
        rows.append(w)

print(f"\nBuilt {len(rows)} windows across {grouped.ngroups} clips.")
if skipped:
    print(f"Skipped {len(skipped)} clips with no valid window:")
    for subject, parent_video, reason in skipped[:10]:
        print(f"   - {subject}/{parent_video}: {reason}")
    if len(skipped) > 10:
        print(f"   ... and {len(skipped) - 10} more")


### Writing the Window Index CSV


In [ ]:
cnn_lstm_windows_csv_path = os.path.join(processed_folder, "cnn_lstm_windows_index.csv")

df_cnn_lstm_windows = pd.DataFrame(rows)
df_cnn_lstm_windows.to_csv(cnn_lstm_windows_csv_path, index=False)

print(f"✅ Window index written: {cnn_lstm_windows_csv_path}")
print(f"   Shape: {df_cnn_lstm_windows.shape}")


### Dataset Sanity Check

Verify class balance across windows, and preview one window's frame sequence to confirm the paths resolve to real, temporally-ordered images.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

df_cnn_lstm_windows = pd.read_csv(cnn_lstm_windows_csv_path)
level_counts = df_cnn_lstm_windows['level'].value_counts().sort_index()

print("--- Dataset Sanity Check ---")
print(f"Total windows: {len(df_cnn_lstm_windows)}")
print(f"Distinct subjects: {df_cnn_lstm_windows['subject'].nunique()}")
print(f"Window duration distribution:\n{df_cnn_lstm_windows['window_duration_sec'].value_counts().sort_index()}")
print("\nWindow counts per level:")
for lvl, count in level_counts.items():
    _nm = CLASS_NAMES[int(lvl) - 1] if 0 <= int(lvl) - 1 < len(CLASS_NAMES) else "?"
    print(f"  Level {lvl} ({_nm}): {count} windows")

_expected = set(range(1, NUM_CLASSES + 1))
_present = {int(x) for x in level_counts.index}
if _present == _expected:
    print(f"\n✅ Success: exactly the {NUM_CLASSES} expected classes {sorted(_expected)} are present.")
else:
    print(f"\n⚠️ Warning: class set {sorted(_present)} != expected {sorted(_expected)} "
          f"(missing {sorted(_expected - _present)}, unexpected {sorted(_present - _expected)}).")

# --- Non-overlap spot check ---
# For each (subject, parent_video, window_duration_sec) group, consecutive windows (sorted by
# start_sample_idx) must not overlap: end_sample_idx of window n < start_sample_idx of window n+1.
# A violation here would mean the tiling logic in build_windows_for_clip regressed back toward
# sliding-window behavior. Written with groupby().shift() rather than groupby().apply() so it
# doesn't depend on a specific pandas version's apply/include_groups behavior.
group_cols = ['subject', 'parent_video', 'window_duration_sec']
df_sorted = df_cnn_lstm_windows.sort_values(group_cols + ['start_sample_idx'])
prev_end_sample_idx = df_sorted.groupby(group_cols)['end_sample_idx'].shift(1)
overlap_found = (prev_end_sample_idx >= df_sorted['start_sample_idx']).fillna(False).any()
print(f"\n{'❌ Overlap detected' if overlap_found else '✅ No overlapping windows within any duration config'}")

# Preview one 20-second window's frame sequence to visually confirm temporal ordering and
# elapsed duration -- printing the real elapsed time explicitly is a cheap, direct way to catch a
# future sampling_fps mismatch (like the one this notebook previously had) on sight, rather than
# it silently producing wrong-duration windows again.
sample = df_cnn_lstm_windows[df_cnn_lstm_windows['window_duration_sec'] == max_context_sec].sample(1, random_state=0).iloc[0]
sample_paths = sample['image_paths'].split(';')
elapsed_sec = sample['n_real_frames'] / sampling_fps
print(f"\nPreviewing a {sample['window_duration_sec']}s window ({len(sample_paths)} frames, "
      f"{elapsed_sec:.1f}s elapsed at {sampling_fps} FPS), subject={sample['subject']}, level={sample['level']}:")

n_preview = min(8, len(sample_paths))
preview_idx = np.linspace(0, len(sample_paths) - 1, n_preview, dtype=int)
fig, axes = plt.subplots(1, n_preview, figsize=(2 * n_preview, 2.5))
for ax, idx in zip(axes, preview_idx):
    ax.imshow(mpimg.imread(sample_paths[idx]))
    ax.set_title(f"t={idx}")
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# --- Geometric feature fusion sanity check ---
geo_seqs = df_cnn_lstm_windows['geometric_feature_seq'].str.split(';')
frame_counts_match = (geo_seqs.str.len() == df_cnn_lstm_windows['n_real_frames']).all()
status_mark = chr(0x2705) if frame_counts_match else chr(0x274c)
print(f"\n{status_mark} geometric_feature_seq frame count matches n_real_frames for every window")

sample_vec = [float(v) for v in geo_seqs.iloc[0][0].split(',')]
print(f"First window's first-frame geometric feature vector ({len(sample_vec)} values, "
      f"order={GEO_FEATURE_NAMES}):\n{sample_vec}")
